# Phase 5: Retrieval Augmented Generation (RAG)

**Goal:** Understand how to give an LLM access to external knowledge without retraining.

You now know:
- How models work (Phase 1)
- How they generate text (Phase 2)
- How prompts steer output (Phase 3)
- How fine-tuning changes the model (Phase 4)

**The problem fine-tuning can't solve:**
- Knowledge changes (new products, updated policies, recent events)
- You can't retrain every time information updates
- The model's knowledge is frozen at training time

**RAG solves this:** Keep the model fixed, but **retrieve relevant documents** and inject them into the prompt at query time.

```
Without RAG:  User question → LLM → answer (from training data only)
With RAG:     User question → Search knowledge base → LLM + relevant docs → answer
```

In [ ]:
import os
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print("Libraries loaded successfully")

---
## 1. The Knowledge Base

Imagine you're building a customer support bot for a tech company. The LLM doesn't know your specific product details, policies, or troubleshooting steps. We need to give it that knowledge.

In production, this would be thousands of documents. Here we use a small set to learn the mechanics.

In [ ]:
# Simulated customer support knowledge base
knowledge_base = [
    {
        "id": "return-policy",
        "text": "Our return policy allows customers to return any product within 30 days of purchase for a full refund. Items must be in original packaging and unused condition. Refunds are processed within 5-7 business days after we receive the returned item. For electronics, the return window is extended to 60 days during the holiday season (November 15 through January 15)."
    },
    {
        "id": "shipping-info",
        "text": "We offer three shipping options: Standard (5-7 business days, free for orders over $50), Express (2-3 business days, $9.99), and Overnight (next business day, $19.99). All orders are shipped from our warehouse in Austin, Texas. International shipping is available to Canada and UK only, with delivery times of 10-14 business days."
    },
    {
        "id": "warranty",
        "text": "All electronics come with a 2-year manufacturer warranty covering defects in materials and workmanship. The warranty does not cover damage from drops, water, or unauthorized modifications. To file a warranty claim, contact support with your order number and a description of the issue. Warranty replacements are typically shipped within 3 business days."
    },
    {
        "id": "account-setup",
        "text": "To create an account, visit our website and click 'Sign Up' in the top right corner. You'll need a valid email address and a password with at least 8 characters including one number and one special character. After registration, you'll receive a verification email. Click the link to activate your account. You can also sign up using Google or Apple ID for faster access."
    },
    {
        "id": "password-reset",
        "text": "To reset your password, click 'Forgot Password' on the login page. Enter the email address associated with your account. You'll receive a password reset link within 5 minutes. The link expires after 24 hours. If you don't receive the email, check your spam folder or contact support. For security, we require password changes every 90 days for business accounts."
    },
    {
        "id": "product-smartwatch-x1",
        "text": "The SmartWatch X1 features a 1.5-inch AMOLED display, heart rate monitoring, GPS tracking, and 5-day battery life. It is water-resistant to 50 meters and compatible with both iOS and Android. The watch supports contactless payments via NFC and has 32GB of internal storage for music. Price: $249.99. Available in black, silver, and rose gold."
    },
    {
        "id": "product-laptop-pro",
        "text": "The LaptopPro 15 comes with a 15.6-inch 4K display, Intel i7 processor, 16GB RAM, and 512GB SSD. Battery life is approximately 10 hours. It weighs 4.2 pounds and has 2 USB-C ports, 1 HDMI port, and a headphone jack. Price: $1,299.99. The Pro model includes a dedicated GPU for gaming and video editing."
    },
    {
        "id": "troubleshoot-wifi",
        "text": "If your device won't connect to WiFi: 1) Restart your device and router. 2) Forget the network in WiFi settings and reconnect. 3) Check that you're within range of the router. 4) Ensure your device software is up to date. 5) Try connecting to a different network to rule out device issues. If the problem persists, contact our support team for a diagnostic session."
    },
    {
        "id": "troubleshoot-battery",
        "text": "For battery drain issues: 1) Check which apps are using the most battery in Settings > Battery. 2) Reduce screen brightness and disable always-on display. 3) Turn off Bluetooth and GPS when not in use. 4) Close background apps. 5) If your device is over 2 years old, the battery may need replacement. Contact support to schedule a battery replacement appointment at $49.99."
    },
    {
        "id": "subscription-plans",
        "text": "We offer three subscription plans: Basic ($9.99/month) includes cloud storage and basic support. Premium ($19.99/month) adds priority support, extended warranty, and exclusive discounts. Enterprise (custom pricing) includes dedicated account manager, SLA guarantees, and bulk ordering. All plans can be cancelled anytime. Annual billing saves 20%."
    },
]

print(f"Knowledge base: {len(knowledge_base)} documents")
for doc in knowledge_base:
    print(f"  [{doc['id']}] {doc['text'][:60]}...")

---
## 2. Step 1 of RAG: Turn Documents into Vectors (Embeddings)

To search documents by *meaning* (not just keywords), we convert each document into a vector — a list of numbers that captures its semantic meaning.

Similar documents will have vectors that are close together in this space.

Remember from Phase 1: embeddings are numerical representations of text. Here we use a **sentence embedding model** that's been trained specifically to put semantically similar texts close together.

In [ ]:
# Load embedding model — this runs locally, no API needed
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed a sample to see what it looks like
sample_text = "How do I return a product?"
sample_embedding = embedding_model.encode(sample_text)

print(f"Text: '{sample_text}'")
print(f"Embedding shape: {sample_embedding.shape}")
print(f"First 10 values: {sample_embedding[:10]}")
print(f"\nEach piece of text becomes a vector of {len(sample_embedding)} numbers.")
print(f"Texts with similar meaning will have similar vectors.")

In [ ]:
# Demonstrate semantic similarity
from sentence_transformers import util

queries = [
    "How do I return a product?",
    "What's your refund policy?",
    "How long does shipping take?",
]

doc_texts = [doc["text"] for doc in knowledge_base]
doc_ids = [doc["id"] for doc in knowledge_base]

query_embeddings = embedding_model.encode(queries)
doc_embeddings = embedding_model.encode(doc_texts)

print("Semantic search — which document is most relevant to each query?\n")
for i, query in enumerate(queries):
    similarities = util.cos_sim(query_embeddings[i], doc_embeddings)[0]
    best_idx = similarities.argmax().item()
    print(f"Query: '{query}'")
    print(f"  Best match: [{doc_ids[best_idx]}] (similarity: {similarities[best_idx]:.4f})")
    
    # Show top 3
    top3 = similarities.argsort(descending=True)[:3]
    for rank, idx in enumerate(top3):
        marker = " ←" if rank == 0 else ""
        print(f"  #{rank+1}: [{doc_ids[idx]}] score={similarities[idx]:.4f}{marker}")
    print()

### YOUR UNDERSTANDING

*Why does "How do I return a product?" match the return-policy document even though the question doesn't use the word "policy"?*

> (your notes here)

*How is this different from keyword search (like Ctrl+F)?*

> (your notes here)


---
## 3. Step 2 of RAG: Store Vectors in a Database

With 10 documents, we could search by brute-force. With millions, we need a **vector database** that indexes embeddings for fast similarity search.

ChromaDB is a lightweight vector database that runs locally — perfect for learning.

In [ ]:
# Create vector database and add documents
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(
    name="customer_support",
    metadata={"hnsw:space": "cosine"}
)

# Add all documents with their embeddings
collection.add(
    ids=[doc["id"] for doc in knowledge_base],
    documents=[doc["text"] for doc in knowledge_base],
    embeddings=embedding_model.encode([doc["text"] for doc in knowledge_base]).tolist(),
)

print(f"Vector database created with {collection.count()} documents")
print(f"Each document stored as text + {sample_embedding.shape[0]}-dimensional vector")

In [ ]:
# Search the vector database

def retrieve(query, n_results=3):
    """Search the vector database for documents relevant to the query."""
    query_embedding = embedding_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
    )
    return results

# Test retrieval
test_queries = [
    "Can I get a refund?",
    "My battery is dying fast",
    "Tell me about the smartwatch",
    "How much is premium support?",
]

for query in test_queries:
    results = retrieve(query, n_results=2)
    print(f"Query: '{query}'")
    for i in range(len(results["ids"][0])):
        doc_id = results["ids"][0][i]
        distance = results["distances"][0][i]
        snippet = results["documents"][0][i][:80]
        print(f"  #{i+1}: [{doc_id}] (distance: {distance:.4f}) {snippet}...")
    print()

---
## 4. Step 3 of RAG: Generate an Answer Using Retrieved Context

Now the key step — combine retrieval with generation:
1. User asks a question
2. We retrieve relevant documents from the vector database
3. We inject those documents into the prompt as context
4. The LLM generates an answer grounded in that context

First, let's see what happens **without** RAG (model relies only on its training data).

In [ ]:
# Load FLAN-T5-base for generation
gen_model_name = "google/flan-t5-base"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_model_name)

def generate_answer(prompt, max_tokens=200):
    """Generate an answer using FLAN-T5."""
    inputs = gen_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    with torch.no_grad():
        output = gen_model.generate(**inputs, max_new_tokens=max_tokens)
    return gen_tokenizer.decode(output[0], skip_special_tokens=True)

# Without RAG — the model has no knowledge of our products/policies
questions = [
    "What is your return policy?",
    "How much does the SmartWatch X1 cost?",
    "My WiFi isn't working on my device, what should I do?",
]

print("WITHOUT RAG — model uses only its training knowledge:\n")
for q in questions:
    answer = generate_answer(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

In [ ]:
# WITH RAG — retrieve context, inject into prompt, then generate

def rag_answer(question, n_context=2):
    """Full RAG pipeline: retrieve + generate."""
    # Step 1: Retrieve relevant documents
    results = retrieve(question, n_results=n_context)
    context_docs = results["documents"][0]
    context_ids = results["ids"][0]
    
    # Step 2: Build prompt with context
    context_text = "\n\n".join(context_docs)
    prompt = (
        f"Answer the customer's question using ONLY the information provided below. "
        f"If the answer is not in the context, say 'I don't have that information.'\n\n"
        f"Context:\n{context_text}\n\n"
        f"Question: {question}\n"
        f"Answer:"
    )
    print(prompt)
    # Step 3: Generate answer
    answer = generate_answer(prompt)
    
    return answer, context_ids, prompt

print("WITH RAG — model uses retrieved documents as context:\n")
for q in questions:
    answer, sources, prompt = rag_answer(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"   Sources: {sources}")
    print()

In [ ]:
# Side-by-side comparison

comparison_questions = [
    "What is your return policy?",
    "How much does the SmartWatch X1 cost?",
    "My WiFi isn't working on my device, what should I do?",
    "What are your shipping options?",
    "How do I reset my password?",
]

print(f"{'Question':<45} {'Without RAG':<35} {'With RAG'}")
print("─" * 130)

for q in comparison_questions:
    no_rag = generate_answer(q)
    with_rag, sources, _ = rag_answer(q)
    
    no_rag_short = no_rag[:32] + "..." if len(no_rag) > 35 else no_rag
    with_rag_short = with_rag[:60] + "..." if len(with_rag) > 63 else with_rag
    q_short = q[:42] + "..." if len(q) > 45 else q
    
    print(f"{q_short:<45} {no_rag_short:<35} {with_rag_short}")

### YOUR UNDERSTANDING

*What's the difference in answer quality with vs without RAG?*

> (your notes here)

*Where does the "knowledge" come from in each case?*

> (your notes here)


---
## 5. See the Full RAG Prompt

RAG isn't magic — it's just prompt engineering with dynamic context injection. Let's look at what the LLM actually receives.

In [ ]:
# Peek inside the RAG prompt
_, _, full_prompt = rag_answer("What is the warranty coverage?")

print("THE ACTUAL PROMPT SENT TO THE LLM:")
print("=" * 60)
print(full_prompt)
print("=" * 60)
print(f"\nPrompt length: {len(full_prompt)} characters")
print(f"\nThe LLM sees your question + the relevant docs.")
print(f"It doesn't 'know' about your products — it reads the docs you gave it.")

---
## 6. RAG with GPT (OpenAI)

FLAN-T5-base is small and limited. Let's see how a large model handles RAG — same retrieval, better generation.


In [ ]:
# Set up OpenAI client
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
gpt_model = "gpt-4o-mini"
print(f"Connected to OpenAI: {gpt_model}")


In [ ]:
# RAG with GPT

def rag_answer_gpt(question, n_context=2):
    """RAG pipeline using OpenAI for generation."""
    # Step 1: Retrieve (same as before — local embedding model)
    results = retrieve(question, n_results=n_context)
    context_docs = results["documents"][0]
    context_ids = results["ids"][0]
    
    # Step 2: Build prompt with context
    context_text = "\n\n".join(context_docs)
    
    # Step 3: Generate with GPT
    response = client.chat.completions.create(
        model=gpt_model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful customer support agent. "
                    "Answer questions using ONLY the provided context. "
                    "If the answer isn't in the context, say you don't have that information. "
                    "Be concise and friendly."
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{context_text}\n\nQuestion: {question}"
            }
        ],
        max_tokens=500,
    )
    
    answer = response.choices[0].message.content
    return answer, context_ids

# Compare FLAN-T5 RAG vs GPT RAG
test_questions = [
    "What is your return policy for electronics?",
    "My smartwatch battery drains fast. What can I do?",
    "I want to upgrade my subscription. What are my options?",
    "Can I ship to Germany?",
]

print("FLAN-T5 RAG vs GPT RAG (same retrieval, different generation)\n")
for q in test_questions:
    flan_answer, flan_sources, _ = rag_answer(q)
    print(f"Q: {q}", flush=True)
    gpt_answer, gpt_sources = rag_answer_gpt(q)
    print(f"  FLAN-T5:  {flan_answer}")
    print(f"  GPT:      {gpt_answer}")
    print(f"  Sources:  {gpt_sources}")
    print()


### YOUR UNDERSTANDING

*Both models received the same retrieved context. Why is GPT's answer better?*

> (your notes here)


---
## 7. RAG Failure Modes: When Retrieval Goes Wrong

RAG is only as good as its retrieval. If the wrong documents are retrieved, the LLM generates a confident but wrong answer. Let's see this in action.

In [ ]:
# Test with questions where retrieval might struggle

tricky_questions = [
    "Do you sell headphones?",              # not in knowledge base at all
    "Can I return a laptop after 45 days?",  # requires combining return policy + product info
    "What's cheaper, express shipping or premium subscription?",  # needs info from 2 docs
]

print("Testing RAG with tricky questions:\n")
for q in tricky_questions:
    results = retrieve(q, n_results=2)
    print(f"Q: {q}")
    print(f"  Retrieved: {results['ids'][0]}")
    print(f"  Distances: {[f'{d:.4f}' for d in results['distances'][0]]}")
    
    answer, sources, _ = rag_answer(q)
    print(f"  FLAN-T5:   {answer}")
    
    print(f"  GPT:     ", end="", flush=True)
    gpt_answer, _ = rag_answer_gpt(q)
    print(f"{gpt_answer}")
    print()

### YOUR UNDERSTANDING

*What happens when the knowledge base doesn't contain the answer? Did the models handle it differently?*

> (your notes here)

*How could you improve retrieval for questions that span multiple documents?*

> (your notes here)


---
## 8. Chunking: Why Document Size Matters

In production, documents are often long (pages of text). You can't embed an entire document as one vector — it loses specificity. The solution is **chunking**: splitting documents into smaller pieces.

In [ ]:
# Demonstrate the chunking concept

long_document = """Customer Support Complete Guide

Chapter 1: Returns and Refunds
Our return policy allows returns within 30 days. Electronics have a 60-day window during holidays. Items must be unused and in original packaging. Refunds process in 5-7 business days.

Chapter 2: Shipping
Standard shipping is free over $50 and takes 5-7 days. Express shipping costs $9.99 for 2-3 day delivery. Overnight is $19.99. International shipping available to Canada and UK only.

Chapter 3: Technical Support
For WiFi issues, restart device and router first. For battery problems, check battery usage in settings. Warranty covers manufacturer defects for 2 years.

Chapter 4: Account Management
Create accounts at our website with email and password. Passwords must be 8+ characters with numbers and special characters. Reset passwords via the Forgot Password link."""

def chunk_by_paragraph(text):
    """Split text into chunks by paragraph."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    return paragraphs

def chunk_by_size(text, chunk_size=200, overlap=50):
    """Split text into fixed-size chunks with overlap."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

# Compare approaches
print("WHOLE DOCUMENT as one embedding:")
print(f"  1 vector for {len(long_document)} characters")
print(f"  Problem: query about 'shipping' matches the whole doc, not shipping section\n")

para_chunks = chunk_by_paragraph(long_document)
print(f"PARAGRAPH chunking: {len(para_chunks)} chunks")
for i, chunk in enumerate(para_chunks):
    print(f"  Chunk {i}: {chunk[:70]}...")

print(f"\nFIXED-SIZE chunking (200 words, 50 overlap): {len(chunk_by_size(long_document))} chunks")

print(f"\nSmaller chunks = more precise retrieval, but less context per chunk")
print(f"Larger chunks = more context, but less precise matching")
print(f"Overlap ensures sentences at chunk boundaries aren't lost")

---
## 9. Summary: The RAG Architecture

```
                    ┌──────────────────────────┐
                    │     Knowledge Base        │
                    │  (documents, FAQs, etc.)  │
                    └────────────┬─────────────┘
                                 │
                    ┌────────────▼─────────────┐
                    │   Embedding Model         │
                    │   (sentence-transformers)  │
                    └────────────┬─────────────┘
                                 │
                    ┌────────────▼─────────────┐
                    │   Vector Database          │
                    │   (ChromaDB / Pinecone)    │
                    └────────────┬─────────────┘
                                 │
    User Question ──► Embed ──► Search ──► Top K docs
                                              │
                                              ▼
                           ┌──────────────────────┐
                           │   Build Prompt:       │
                           │   System instruction  │
                           │   + Retrieved docs    │
                           │   + User question     │
                           └──────────┬───────────┘
                                      │
                                      ▼
                           ┌──────────────────────┐
                           │   LLM (generation)    │
                           │   FLAN-T5 / GPT     │
                           └──────────┬───────────┘
                                      │
                                      ▼
                              Grounded Answer
```

### Key Components:
| Component | Role | Example from this notebook |
|---|---|---|
| Knowledge Base | Source of truth | 10 customer support docs |
| Embedding Model | Convert text → vectors | all-MiniLM-L6-v2 |
| Vector Database | Fast similarity search | ChromaDB |
| Retriever | Find relevant docs for a query | cosine similarity search |
| Generator (LLM) | Produce answer from context | FLAN-T5 / GPT |

### RAG vs Fine-Tuning:
| | RAG | Fine-Tuning |
|---|---|---|
| Updates knowledge | Add/update documents (instant) | Retrain the model (hours/days) |
| Source of truth | External documents (auditable) | Baked into model weights (opaque) |
| Hallucination risk | Lower — answer grounded in docs | Higher — model may confabulate |
| Best for | Factual, changing info | Style, behavior, format changes |

### YOUR UNDERSTANDING — Final Reflection

*In your own words, what is RAG and why is it useful?*

> (your notes here)

*When would you use RAG vs fine-tuning vs just prompting?*

> (your notes here)

*What could go wrong with RAG in production?*

> (your notes here)


---
## What You've Learned Across All 5 Phases

| Phase | What | Key Insight |
|---|---|---|
| 1. Foundations | Transformers, attention, architectures | Models learn word relationships through self-attention |
| 2. Text Generation | Greedy, beam, sampling, temperature | You control randomness vs determinism in output |
| 3. Prompt Engineering | Zero/few-shot, chain-of-thought | Steer the model without changing it |
| 4. Fine-Tuning | Full FT vs LoRA/PEFT | Teach the model new tasks with minimal parameters |
| 5. RAG | Retrieval + generation | Give the model external knowledge at query time |

**You now have the complete foundation for building the AI Customer Support project.**